# 🍅 YOLO TOMATO DETECTOR TRAINING

**Simple 5-step process:**
1. Change runtime to GPU (faster training!)
2. Run Cell 1: Install YOLO
3. Run Cell 2: Upload your data.zip
4. Run Cell 3: Train model (1-2 hours - automated!)
5. Run Cell 4: Download trained model

**Time:** 1-2 hours (mostly waiting)

**Result:** tomato_detector.pt (use this on your Raspberry Pi!)

## ⚙️ FIRST: Change Runtime to GPU

**IMPORTANT - Do this first for faster training!**

1. Click: **Runtime** (top menu)
2. Click: **Change runtime type**
3. Hardware accelerator: Select **GPU** (T4)
4. Click: **Save**

✅ GPU enabled = Training 10x faster!

## 📦 CELL 1: Install YOLO (Ultralytics)

**Click the ▶ play button**

This installs YOLOv8 (takes 1-2 minutes)

In [ ]:
print("🔧 Installing YOLOv8 (Ultralytics)...")
print("This takes 1-2 minutes. Please wait...\n")

!pip install -q ultralytics

from ultralytics import YOLO
import os
import shutil
from google.colab import files

print("\n✅ YOLOv8 installed successfully!")
print("\n👉 Next: Run CELL 2 to upload your data.zip")

## 📤 CELL 2: Upload Your data.zip

**Click the ▶ play button**

A file upload dialog will appear. Select your **data.zip** file.

In [ ]:
import zipfile

print("📤 Upload your data.zip file")
print("Click 'Choose Files' and select data.zip\n")

# Upload ZIP
uploaded = files.upload()

# Extract ZIP
print("\n📦 Extracting data.zip...\n")
with zipfile.ZipFile('data.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("✅ Data extracted!\n")

# Check structure
print("📂 Dataset structure:")
!ls -la /content/dataset

# Count images and labels
images_dir = '/content/dataset/images'
labels_dir = '/content/dataset/labels'

if os.path.exists(images_dir):
    image_count = len([f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
    print(f"\n✅ Found {image_count} images")

if os.path.exists(labels_dir):
    label_count = len([f for f in os.listdir(labels_dir) if f.endswith('.txt') and f != 'classes.txt'])
    print(f"✅ Found {label_count} label files")

# Read classes
classes_file = '/content/dataset/labels/classes.txt'
if os.path.exists(classes_file):
    with open(classes_file, 'r') as f:
        classes = f.read().strip().split('\n')
    print(f"✅ Classes: {classes}")
else:
    # Try alternate location
    classes_file = '/content/dataset/classes.txt'
    if os.path.exists(classes_file):
        with open(classes_file, 'r') as f:
            classes = f.read().strip().split('\n')
        print(f"✅ Classes: {classes}")

print("\n👉 Next: Run CELL 3 to prepare dataset and start training")

## 🎯 CELL 3: Prepare Dataset & Train YOLO

**Click the ▶ play button**

This will:
1. Organize your data into train/val splits (80/20)
2. Create data.yaml configuration
3. Train YOLOv8 model
4. Show training progress

**Training takes 1-2 hours - be patient!**

☕ **Tip:** Go get coffee, work on something else, check back in 1-2 hours!

In [ ]:
import random
import yaml

print("📋 Preparing dataset for YOLO training...\n")

# Create directory structure
os.makedirs('/content/yolo_dataset/images/train', exist_ok=True)
os.makedirs('/content/yolo_dataset/images/val', exist_ok=True)
os.makedirs('/content/yolo_dataset/labels/train', exist_ok=True)
os.makedirs('/content/yolo_dataset/labels/val', exist_ok=True)

# Get all image files
images_dir = '/content/dataset/images'
labels_dir = '/content/dataset/labels'

image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
random.shuffle(image_files)

# Split 80/20 train/val
split_idx = int(len(image_files) * 0.8)
train_images = image_files[:split_idx]
val_images = image_files[split_idx:]

print(f"📊 Dataset split:")
print(f"   Training: {len(train_images)} images")
print(f"   Validation: {len(val_images)} images\n")

# Copy train files
print("📁 Copying training files...")
for img_file in train_images:
    # Copy image
    shutil.copy(
        os.path.join(images_dir, img_file),
        f'/content/yolo_dataset/images/train/{img_file}'
    )
    # Copy label
    label_file = os.path.splitext(img_file)[0] + '.txt'
    label_path = os.path.join(labels_dir, label_file)
    if os.path.exists(label_path):
        shutil.copy(
            label_path,
            f'/content/yolo_dataset/labels/train/{label_file}'
        )

# Copy val files
print("📁 Copying validation files...")
for img_file in val_images:
    # Copy image
    shutil.copy(
        os.path.join(images_dir, img_file),
        f'/content/yolo_dataset/images/val/{img_file}'
    )
    # Copy label
    label_file = os.path.splitext(img_file)[0] + '.txt'
    label_path = os.path.join(labels_dir, label_file)
    if os.path.exists(label_path):
        shutil.copy(
            label_path,
            f'/content/yolo_dataset/labels/val/{label_file}'
        )

print("\n✅ Dataset prepared!\n")

# Read classes
classes_file = '/content/dataset/labels/classes.txt'
if not os.path.exists(classes_file):
    classes_file = '/content/dataset/classes.txt'

with open(classes_file, 'r') as f:
    classes = f.read().strip().split('\n')

# Create data.yaml
data_yaml = {
    'path': '/content/yolo_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(classes),
    'names': classes
}

with open('/content/yolo_dataset/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("📄 Created data.yaml:")
print(yaml.dump(data_yaml, default_flow_style=False))

print("\n" + "="*60)
print("🚀 STARTING YOLO TRAINING")
print("="*60)
print("\nThis will take 1-2 hours...")
print("You can:")
print("  ☕ Get coffee")
print("  💼 Work on something else")
print("  📱 Check your phone")
print("\nTraining will continue automatically!")
print("Just keep this tab open.\n")
print("="*60 + "\n")

# Train YOLO model
model = YOLO('yolov8n.pt')  # Use nano model (fastest, good for Raspberry Pi)

results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='tomato_detector',
    patience=20,
    save=True,
    device=0  # Use GPU
)

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print("\n✅ Model trained successfully!")
print("\n👉 Next: Run CELL 4 to download your trained model")
print("="*60)

## 📥 CELL 4: Download Trained Model

**After training completes:**

Run this cell to download your trained model!

In [ ]:
print("📥 Preparing model for download...\n")

# Find the trained model
model_path = '/content/runs/detect/tomato_detector/weights/best.pt'

if os.path.exists(model_path):
    # Copy to easier location
    shutil.copy(model_path, '/content/tomato_detector.pt')
    
    print("✅ Model ready!\n")
    
    # Show training results
    print("📊 Training Results:")
    results_img = '/content/runs/detect/tomato_detector/results.png'
    if os.path.exists(results_img):
        from IPython.display import Image, display
        print("\nTraining curves:")
        display(Image(filename=results_img))
    
    # Show confusion matrix
    confusion_img = '/content/runs/detect/tomato_detector/confusion_matrix.png'
    if os.path.exists(confusion_img):
        print("\nConfusion Matrix:")
        display(Image(filename=confusion_img))
    
    print("\n" + "="*60)
    print("📥 DOWNLOADING MODEL...")
    print("="*60 + "\n")
    
    # Download model
    files.download('/content/tomato_detector.pt')
    
    print("\n" + "="*60)
    print("🎉 SUCCESS!")
    print("="*60)
    print("\nYou now have: tomato_detector.pt")
    print("\n📋 Next Steps:")
    print("   1. Save tomato_detector.pt file")
    print("   2. Copy it to your Raspberry Pi")
    print("   3. Update your camera script to use YOLO")
    print("   4. Run detection - only detects tomatoes! ✅")
    print("\n👉 Tell me: 'Model trained! Ready to deploy to Pi!'")
    print("="*60)
    
else:
    print("❌ Model not found. Make sure training completed successfully.")
    print(f"Looking for: {model_path}")
    print("\nTry running CELL 3 again.")

---

## 🎯 SUMMARY

**What you did:**
1. ✅ Installed YOLOv8
2. ✅ Uploaded your labeled dataset (data.zip)
3. ✅ Trained YOLO model (1-2 hours)
4. ✅ Downloaded tomato_detector.pt

**Next step:**
- Copy tomato_detector.pt to Raspberry Pi
- Update camera script to use YOLO
- Only detects actual tomatoes! ✅
- Ignores walls, shirts, curtains! ✅

**Expected accuracy:** 80-90% (with 40 images)

**Want better accuracy?**
- Take 40-60 more photos
- Label them
- Retrain with 80-100 total images
- Get 90-95% accuracy!

---

## 🚀 CONGRATULATIONS!

You just trained a custom YOLO model! 🎉

---